# EcoAgentOps Experiments Notebook

This notebook implements the experimental protocol for EcoAgentOps research paper, updated for March 2026.

## Overview

EcoAgentOps is a framework for carbon-aware dataset pruning and model training in multimodal AI systems. This notebook guides through the complete experimental pipeline including environment setup, data preparation, surrogate model training, RL policy optimization, and full pipeline evaluation.

## Phases

1. **Phase 0**: Environment & Tools Setup
2. **Phase 1**: Datasets – Download & Prepare  
3. **Phase 2**: Surrogate Training
4. **Phase 3**: RL Policy Training (PPO)
5. **Phase 4**: Full Pipeline Runs – Baselines + Ours
6. **Phase 5-7**: Ablations, Sensitivity, Deployment, Stats

**Hardware Requirements**:
- GPU: A100 40/80GB equivalent
- RAM: 64GB+
- Storage: 1TB+
- Network: Stable for downloads

**Estimated Time**: 10-14 days
**Carbon Context**: Delhi grid ~0.70-0.72 kgCO₂/kWh (2026)

## Phase 0: Environment & Tools Setup

This phase sets up the conda environment, installs all required packages, and configures tools like WandB and CodeCarbon for carbon tracking.

In [ ]:
# Phase 1: Datasets — Download & prepare (run in terminal for large sizes)

# Minimal in-notebook example to stream metadata and save preview URLs.
from datasets import load_dataset
import pandas as pd
import os

out_dir = 'data/laion_1M'
os.makedirs(out_dir, exist_ok=True)

print('Streaming 10000 samples as a quick test (increase for full run)')
ds = load_dataset('laion/relaion2B-en-research-safe', split='train', streaming=True)

samples = []
for i, ex in enumerate(ds):
    samples.append({'URL': ex.get('URL'), 'TEXT': ex.get('TEXT'), 'id': i})
    if (i + 1) >= 10000:
        break

import random
random.seed(42)
random.shuffle(samples)

pd.DataFrame(samples).to_parquet(os.path.join(out_dir, 'metadata_preview.parquet'))
urls = [s['URL'] for s in samples if s['URL']][:10000]
with open(os.path.join(out_dir, 'urls_preview.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(urls))

print('Wrote metadata_preview.parquet and urls_preview.txt in', out_dir)

# To download images with img2dataset (run in terminal):
# img2dataset --url_list data/laion_1M/urls_preview.txt --output_folder data/laion_1M/images --processes_count 16 --thread_count 256 --image_size 224 --resize_mode "keep_ratio" --format webdataset
